# Save Best-Fold Models for Each Language Condition

Retrains the best fold for each condition and saves:
- Model head weights (state_dict)
- Full config (architecture, speakers, fold, epoch)

**Best folds from previous experiments:**
- English: fold 1 (val speakers 17, 18) — 86.49%
- Chinese: fold 2 (val speakers 3, 9) — 81.51%
- Combined: fold 1 (val speakers 18, 20, 7, 9) — 79.11%


In [1]:
!pip install -q soundfile librosa resampy transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 38.9 MB/s eta 0:00:00


In [2]:
import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import soundfile as sf
import librosa
from tqdm import tqdm

from transformers import AutoFeatureExtractor, AutoModelForAudioClassification, get_cosine_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


Device: cuda


## 1. Mount Drive


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 2. Config


In [4]:
# ---- Paths ----
ESD_GT_CSV = Path("/content/drive/MyDrive/ESD/ESD_GT_Colabo.csv")
ESD_ZIP_PATH = Path("/content/drive/MyDrive/ESD_ZIP.zip")
ESD_AUDIO_ROOT = Path("/content/drive/MyDrive/ESD")
LOCAL_CACHE_ESD = Path("/content/audio_cache_esd")

SAVE_ROOT = Path("/content/drive/MyDrive/saved_models")

MODEL_ID = "ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition"
TARGET_SR = 16_000
SHARED_EMOTIONS = sorted(["angry", "happy", "neutral", "sad", "surprised"])

# Training
BATCH_SIZE = 32
CHUNK_ROWS = 200
NUM_EPOCHS = 15
LR = 3e-4
WEIGHT_DECAY = 1e-2
WARMUP_FRAC = 0.10
MAX_GRAD_NORM = 1.0
LABEL_SMOOTH = 0.05
PATIENCE = 4
PROJ_DIM = 256
HIDDEN_DIM = 128
DROPOUT = 0.1

# ---- Best folds to retrain ----
BEST_FOLDS = {
    "english": {
        "train_speakers": ["14", "13", "19", "16", "20", "15", "11", "12"],
        "val_speakers": ["17", "18"],
        "best_epoch": 7,
        "best_acc": 0.8649,
    },
    "chinese": {
        "train_speakers": ["7", "2", "8", "5", "6", "4", "1", "10"],
        "val_speakers": ["3", "9"],
        "best_epoch": 10,
        "best_acc": 0.8151,
    },
    "all": {
        "train_speakers": ["16", "11", "13", "15", "12", "17", "14", "19",
                           "8", "3", "1", "2", "10", "4", "5", "6"],
        "val_speakers": ["18", "20", "7", "9"],
        "best_epoch": 13,
        "best_acc": 0.7911,
    },
}

print("Config OK")
for cond, info in BEST_FOLDS.items():
    print(f"  {cond}: train={len(info['train_speakers'])} spk, "
          f"val={info['val_speakers']}, target_epoch={info['best_epoch']}")


Config OK
  english: train=8 spk, val=['17', '18'], target_epoch=7
  chinese: train=8 spk, val=['3', '9'], target_epoch=10
  all: train=16 spk, val=['18', '20', '7', '9'], target_epoch=13


## 3. Extract audio


In [5]:
import subprocess

LOCAL_CACHE_ESD.mkdir(exist_ok=True)
existing = list(LOCAL_CACHE_ESD.rglob("*.wav"))
if len(existing) > 1000:
    print(f"ESD cache: {len(existing)} files")
else:
    if ESD_ZIP_PATH.exists():
        print("Extracting...")
        subprocess.run(['unzip', '-q', '-o', str(ESD_ZIP_PATH), '-d', str(LOCAL_CACHE_ESD)],
                       capture_output=True, text=True, check=True)
        print(f"Extracted {len(list(LOCAL_CACHE_ESD.rglob('*.wav')))} files")
    else:
        LOCAL_CACHE_ESD = ESD_AUDIO_ROOT


Extracting...
Extracted 35000 files


## 4. Load GT


In [6]:
esd_path_map = {f.name: str(f) for f in LOCAL_CACHE_ESD.rglob("*.wav")}

esd_df = pd.read_csv(ESD_GT_CSV, sep=";")
esd_df["emotion"] = esd_df["emotion"].astype(str).str.strip().str.lower()
esd_df["emotion"] = esd_df["emotion"].replace({"surprise": "surprised"})

if "speaker" in esd_df.columns:
    esd_df["speaker"] = esd_df["speaker"].astype(str).str.strip()
elif "speaker_id" in esd_df.columns:
    esd_df["speaker"] = esd_df["speaker_id"].astype(str).str.strip()

if "filename" in esd_df.columns:
    esd_df["local_path"] = esd_df["filename"].astype(str).str.strip().map(esd_path_map)
elif "full_path" in esd_df.columns:
    esd_df["local_path"] = esd_df["full_path"].apply(
        lambda p: esd_path_map.get(os.path.basename(str(p).strip()), ""))

esd_df = esd_df[esd_df["emotion"].isin(SHARED_EMOTIONS)].copy()
esd_df = esd_df[esd_df["local_path"].notna() & (esd_df["local_path"] != "")].copy()
esd_df = esd_df[esd_df["local_path"].apply(lambda p: os.path.exists(str(p)))].copy()
esd_df = esd_df.reset_index(drop=True)

labels_sorted = SHARED_EMOTIONS
label2id = {lab: i for i, lab in enumerate(labels_sorted)}
num_labels = len(labels_sorted)
esd_df["y"] = esd_df["emotion"].map(label2id).astype(int)

print(f"Total samples: {len(esd_df)}")
print(f"Speakers: {sorted(esd_df['speaker'].unique())}")


Total samples: 34994
Speakers: ['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '3', '4', '5', '6', '7', '8', '9']


## 5. Audio + model helpers


In [7]:
def load_audio_mono_resample(path, target_sr=TARGET_SR):
    wav, sr = sf.read(path, dtype='float32')
    if wav.size == 0:
        raise RuntimeError(f"Empty: {path}")
    if len(wav.shape) > 1:
        wav = wav.mean(axis=1)
    if sr != target_sr:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=target_sr, res_type='soxr_hq')
    return wav

def chunk_rows(rows, n):
    for i in range(0, len(rows), n):
        yield rows[i:i+n]

def load_audio_batch_sequential(paths):
    results, valid_indices = [], []
    for i, p in enumerate(paths):
        try:
            results.append(torch.from_numpy(load_audio_mono_resample(p)).float())
            valid_indices.append(i)
        except:
            pass
    return results, valid_indices

extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)

def forward_batch(model, waveforms, labels):
    inputs = extractor(
        [w.numpy() for w in waveforms],
        sampling_rate=TARGET_SR, return_tensors="pt", padding=True,
    )
    inputs = {k: v.to(device, non_blocking=True) for k, v in inputs.items()}
    y = torch.tensor(labels, dtype=torch.long, device=device)
    logits = model(**inputs)
    return logits, y

print("Helpers ready")


preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

Helpers ready


## 6. Architecture


In [8]:
class LNMLPHead(nn.Module):
    def __init__(self, in_features, num_labels, proj_dim=256, hidden_dim=128, dropout=0.1):
        super().__init__()
        self.input_ln = nn.LayerNorm(in_features)
        self.proj = nn.Linear(in_features, proj_dim)
        self.ln = nn.LayerNorm(proj_dim)
        self.mlp = nn.Sequential(
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(proj_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_labels),
        )
        nn.init.xavier_uniform_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)
        for m in self.mlp.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.input_ln(x)
        x = self.proj(x)
        x = self.ln(x)
        return self.mlp(x)


class HeadOnlyModel(nn.Module):
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head = head
        for p in self.backbone.parameters():
            p.requires_grad = False

    def forward(self, **inputs):
        with torch.no_grad():
            hidden = self.backbone(**inputs).last_hidden_state
            embeddings = hidden.mean(dim=1)
        return self.head(embeddings)

print("Architecture defined")


Architecture defined


## 7. Train and save each condition


In [9]:
@torch.inference_mode()
def evaluate(model, rows, criterion):
    model.eval()
    total, correct, total_loss = 0, 0, 0.0
    all_true, all_pred = [], []
    buf_w, buf_y = [], []

    for chunk in chunk_rows(rows, CHUNK_ROWS):
        paths = [p for p, _ in chunk]
        labels = [y for _, y in chunk]
        wavs, valid_idx = load_audio_batch_sequential(paths)
        valid_labels = [labels[i] for i in valid_idx]

        for w, y in zip(wavs, valid_labels):
            buf_w.append(w)
            buf_y.append(int(y))
            if len(buf_w) >= BATCH_SIZE:
                logits, y_t = forward_batch(model, buf_w, buf_y)
                loss = criterion(logits, y_t)
                preds = torch.argmax(logits, dim=-1)
                total_loss += loss.item() * len(buf_w)
                correct += (preds == y_t).sum().item()
                total += len(buf_w)
                all_true.extend(y_t.cpu().tolist())
                all_pred.extend(preds.cpu().tolist())
                buf_w, buf_y = [], []

    if buf_w:
        logits, y_t = forward_batch(model, buf_w, buf_y)
        loss = criterion(logits, y_t)
        preds = torch.argmax(logits, dim=-1)
        total_loss += loss.item() * len(buf_w)
        correct += (preds == y_t).sum().item()
        total += len(buf_w)
        all_true.extend(y_t.cpu().tolist())
        all_pred.extend(preds.cpu().tolist())

    if total == 0:
        return 0, 0
    acc = correct / total
    f1 = f1_score(all_true, all_pred, average="macro", zero_division=0)
    return acc, f1


def train_and_save(condition_name, train_spk, val_spk, target_epoch):
    print(f"\n{'='*60}")
    print(f"  CONDITION: {condition_name}")
    print(f"  Train: {train_spk}")
    print(f"  Val:   {val_spk}")
    print(f"  Target epoch: {target_epoch}")
    print(f"{'='*60}")

    train_df = esd_df[esd_df["speaker"].isin(train_spk)]
    val_df = esd_df[esd_df["speaker"].isin(val_spk)]
    train_rows = list(zip(train_df["local_path"].tolist(), train_df["y"].tolist()))
    val_rows = list(zip(val_df["local_path"].tolist(), val_df["y"].tolist()))
    print(f"  Train: {len(train_rows)}, Val: {len(val_rows)}")

    # Class balancing
    by_class = {}
    for p, y in train_rows:
        by_class.setdefault(int(y), []).append((p, int(y)))
    for c in by_class:
        random.shuffle(by_class[c])

    # Build model
    base_model = AutoModelForAudioClassification.from_pretrained(MODEL_ID)
    backbone = base_model.wav2vec2
    embed_dim = base_model.config.hidden_size

    head = LNMLPHead(embed_dim, num_labels, PROJ_DIM, HIDDEN_DIM, DROPOUT)
    model = HeadOnlyModel(backbone, head)
    model.to(device)

    optimizer = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)

    steps_per_epoch = max(1, len(train_rows) // BATCH_SIZE)
    total_steps = max(1, steps_per_epoch * NUM_EPOCHS)
    warmup_steps = max(1, int(WARMUP_FRAC * total_steps))
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    best_val_acc = -1.0
    best_state = None
    best_epoch = 0
    no_improve = 0

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        running_loss, total_samples, correct = 0.0, 0, 0

        cls_keys = sorted(by_class.keys())
        max_len = max(len(by_class[c]) for c in cls_keys)
        epoch_rows = []
        for i in range(max_len):
            for c in cls_keys:
                if i < len(by_class[c]):
                    epoch_rows.append(by_class[c][i])
        random.shuffle(epoch_rows)

        buf_w, buf_y = [], []
        for chunk in chunk_rows(epoch_rows, CHUNK_ROWS):
            paths = [p for p, _ in chunk]
            labels_chunk = [y for _, y in chunk]
            wavs, valid_idx = load_audio_batch_sequential(paths)
            valid_labels = [labels_chunk[i] for i in valid_idx]

            for w, y in zip(wavs, valid_labels):
                buf_w.append(w)
                buf_y.append(int(y))
                if len(buf_w) >= BATCH_SIZE:
                    optimizer.zero_grad(set_to_none=True)
                    logits, y_t = forward_batch(model, buf_w, buf_y)
                    loss = criterion(logits, y_t)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(head.parameters(), MAX_GRAD_NORM)
                    optimizer.step()
                    scheduler.step()
                    bs = len(buf_w)
                    running_loss += loss.item() * bs
                    preds = torch.argmax(logits, dim=-1)
                    correct += (preds == y_t).sum().item()
                    total_samples += bs
                    buf_w, buf_y = [], []

        if buf_w:
            optimizer.zero_grad(set_to_none=True)
            logits, y_t = forward_batch(model, buf_w, buf_y)
            loss = criterion(logits, y_t)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(head.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            bs = len(buf_w)
            running_loss += loss.item() * bs
            preds = torch.argmax(logits, dim=-1)
            correct += (preds == y_t).sum().item()
            total_samples += bs

        train_acc = correct / total_samples if total_samples > 0 else 0
        train_loss = running_loss / total_samples if total_samples > 0 else 0

        val_acc, val_f1 = evaluate(model, val_rows, criterion)

        print(f"  Epoch {epoch}/{NUM_EPOCHS}: train_acc={train_acc:.4f} loss={train_loss:.4f} | "
              f"val_acc={val_acc:.4f} f1={val_f1:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            best_state = {k: v.cpu().clone() for k, v in head.state_dict().items()}
            no_improve = 0
            print(f"    >>> New best! val_acc={best_val_acc:.4f}")
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f"    Early stopping at epoch {epoch}")
                break

    # Save
    save_dir = SAVE_ROOT / condition_name
    save_dir.mkdir(parents=True, exist_ok=True)

    # Save head weights
    weights_path = save_dir / "head_weights.pt"
    torch.save(best_state, str(weights_path))
    print(f"  Saved head weights: {weights_path}")

    # Save config
    config = {
        "condition": condition_name,
        "model_id": MODEL_ID,
        "embed_dim": embed_dim,
        "num_labels": num_labels,
        "labels": labels_sorted,
        "label2id": label2id,
        "proj_dim": PROJ_DIM,
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "input_layernorm": True,
        "train_speakers": train_spk,
        "val_speakers": val_spk,
        "best_epoch": best_epoch,
        "best_val_acc": float(best_val_acc),
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "label_smoothing": LABEL_SMOOTH,
        "seed": SEED,
    }
    config_path = save_dir / "config.json"
    with open(config_path, "w") as f:
        json.dump(config, f, indent=2)
    print(f"  Saved config: {config_path}")

    print(f"\n  {condition_name} DONE: best_acc={best_val_acc:.4f} at epoch {best_epoch}")

    del model, backbone, optimizer, scheduler
    torch.cuda.empty_cache()

    return best_val_acc, best_epoch

print("train_and_save() defined")


train_and_save() defined


## 8. Run all three conditions


In [10]:
results = {}

for cond_name, info in BEST_FOLDS.items():
    acc, ep = train_and_save(
        condition_name=cond_name,
        train_spk=info["train_speakers"],
        val_spk=info["val_speakers"],
        target_epoch=info["best_epoch"],
    )
    results[cond_name] = {"acc": acc, "epoch": ep}

print(f"\n{'='*60}")
print(f"  SUMMARY")
print(f"{'='*60}")
for cond, r in results.items():
    print(f"  {cond:<10s}: acc={r['acc']:.4f} epoch={r['epoch']}")
    print(f"    Saved to: {SAVE_ROOT / cond}")
print(f"{'='*60}")



  CONDITION: english
  Train: ['14', '13', '19', '16', '20', '15', '11', '12']
  Val:   ['17', '18']
  Target epoch: 7
  Train: 14000, Val: 3494


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition
Key                      | Status     | 
-------------------------+------------+-
classifier.output.bias   | UNEXPECTED | 
classifier.dense.weight  | UNEXPECTED | 
classifier.output.weight | UNEXPECTED | 
classifier.dense.bias    | UNEXPECTED | 
classifier.bias          | MISSING    | 
classifier.weight        | MISSING    | 
projector.bias           | MISSING    | 
projector.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Wav2Vec2ForSequenceClassification LOAD REPORT from: ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition
Key                      | Status     | 
-------------------------+------------+-
classifier.output.bias   | UNEXPECTED | 
classifier.dense.weight  | UNEXPECTED | 
classifier.output.weight | UNEXPECTED | 
classifier.dense.bias    | UNEXPECTED | 
classifier.bias          | MISSING    | 
classifier.weight        | MISSING    | 
projector.bias           | MISSING    | 
projector.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Epoch 1/15: train_acc=0.6181 loss=0.9964 | val_acc=0.6297 f1=0.6138
    >>> New best! val_acc=0.6297
  Epoch 2/15: train_acc=0.7241 loss=0.7865 | val_acc=0.6514 f1=0.6350
    >>> New best! val_acc=0.6514
  Epoch 3/15: train_acc=0.7622 loss=0.7096 | val_acc=0.7234 f1=0.7191
    >>> New best! val_acc=0.7234
  Epoch 4/15: train_acc=0.7852 loss=0.6672 | val_acc=0.7589 f1=0.7574
    >>> New best! val_acc=0.7589
  Epoch 5/15: train_acc=0.8064 loss=0.6373 | val_acc=0.7394 f1=0.7305
  Epoch 6/15: train_acc=0.8121 loss=0.6141 | val_acc=0.7391 f1=0.7376
  Epoch 7/15: train_acc=0.8282 loss=0.5884 | val_acc=0.7857 f1=0.7809
    >>> New best! val_acc=0.7857
  Epoch 8/15: train_acc=0.8416 loss=0.5679 | val_acc=0.7900 f1=0.7868
    >>> New best! val_acc=0.7900
  Epoch 9/15: train_acc=0.8457 loss=0.5532 | val_acc=0.7714 f1=0.7634
  Epoch 10/15: train_acc=0.8534 loss=0.5380 | val_acc=0.8063 f1=0.8023
    >>> New best! val_acc=0.8063
  Epoch 11/15: train_acc=0.8681 loss=0.5182 | val_acc=0.8057 f1=0.80

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition
Key                      | Status     | 
-------------------------+------------+-
classifier.output.bias   | UNEXPECTED | 
classifier.dense.weight  | UNEXPECTED | 
classifier.output.weight | UNEXPECTED | 
classifier.dense.bias    | UNEXPECTED | 
classifier.bias          | MISSING    | 
classifier.weight        | MISSING    | 
projector.bias           | MISSING    | 
projector.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Epoch 1/15: train_acc=0.6155 loss=1.0199 | val_acc=0.6922 f1=0.6950
    >>> New best! val_acc=0.6922
  Epoch 2/15: train_acc=0.7166 loss=0.8191 | val_acc=0.7452 f1=0.7526
    >>> New best! val_acc=0.7452
  Epoch 3/15: train_acc=0.7483 loss=0.7589 | val_acc=0.7762 f1=0.7739
    >>> New best! val_acc=0.7762
  Epoch 4/15: train_acc=0.7641 loss=0.7227 | val_acc=0.7728 f1=0.7738
  Epoch 5/15: train_acc=0.7762 loss=0.6948 | val_acc=0.7840 f1=0.7856
    >>> New best! val_acc=0.7840
  Epoch 6/15: train_acc=0.7897 loss=0.6695 | val_acc=0.7908 f1=0.7938
    >>> New best! val_acc=0.7908
  Epoch 7/15: train_acc=0.7986 loss=0.6523 | val_acc=0.7647 f1=0.7739
  Epoch 8/15: train_acc=0.8124 loss=0.6302 | val_acc=0.7970 f1=0.7993
    >>> New best! val_acc=0.7970
  Epoch 9/15: train_acc=0.8163 loss=0.6177 | val_acc=0.7933 f1=0.7974
  Epoch 10/15: train_acc=0.8248 loss=0.5997 | val_acc=0.7758 f1=0.7824
  Epoch 11/15: train_acc=0.8380 loss=0.5813 | val_acc=0.7993 f1=0.8038
    >>> New best! val_acc=0.79

## 9. How to load saved models later

```python
import torch
import json

# Load config
with open("saved_models/english/config.json") as f:
    config = json.load(f)

# Rebuild head
head = LNMLPHead(
    config["embed_dim"], config["num_labels"],
    config["proj_dim"], config["hidden_dim"], config["dropout"]
)

# Load weights
state = torch.load("saved_models/english/head_weights.pt", map_location="cpu")
head.load_state_dict(state)
head.eval()

# Combine with backbone
base_model = AutoModelForAudioClassification.from_pretrained(config["model_id"])
backbone = base_model.wav2vec2
model = HeadOnlyModel(backbone, head)
model.eval()
```
